## Installing Dependencies

In [1]:
print("Installing Dependencies\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken

print('Installation completed')

Installing Dependencies

Installation completed



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Model Configuration Variables

In [1]:
LLM_Provider = 'groq'
LLM_Model = 'llama-3.3-70b-versatile'  # In the screenshot this was 'openai/gpt-oss-20b'

Environment Setup & API Key Check

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("GROQ_API_KEY"):
    print("GROQ API KEY is imported!")
else:
    print("GROQ API KEY is not found!")

GROQ API KEY is imported!


Load Documents

In [3]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS_PATH = "./zyro-dynamics-hr-corpus/"  # <-- Knowledge Base
loader = PyPDFDirectoryLoader(CORPUS_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} documents")

C:\Users\Dell\AppData\Local\Temp\ipykernel_11428\3721277009.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 39 documents


Split Documents into Chunks

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 107 chunks


Embedding Model Initialization

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings()
print("Embeddings done!")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1671.79it/s]


Embeddings done!


Vector Store & Retriever

In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)

# Retrieval
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Done!")

Done!


LLM Initialization

In [ ]:
from langchain_groq import ChatGroq

llm_model = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0.7,
    max_tokens = 500
)

print(f"LLM Model groq initialized")

LLM Model groq initialized


RAG Chain Setup

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are an HR assistant. Answer the question using ONLY
the context below. If the answer isn't in the context,
say you don't have that information.

context : {context},
Question : {question}"""
)

def format_docs(docs):
    return ".\n\n".join(d.page_content for d in docs)

@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = RAG_PROMPT | llm_model | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    return {"answer": answer, "sources": docs}

Guardrails

In [13]:
GUARDIAL_PROMPT = ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant. Decide whether the question
below is something an HR assistant should answer (company leave policy,
reimbursement, code of conduct, or other internal HR topics) or something
out of scope (general knowledge, coding help, unrelated small talk, etc).
Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.
Question: {question}
""")

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)

def ask_bot(question: str):
    guardial_chain = GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict = guardial_chain.invoke({"question": question}).strip().upper()
    if 'OUT_OF_SCOPE' in verdict:
        return {'answer': REFUSAL_MESSAGE, "sources": []}
    return rag_chain(question)

print("Guardial Initialized!")

Guardial Initialized!


Testing the Bot

In [14]:
test_questions = [
    "How many casual leaves do I get per year?",
    "What is the internet reimbursement limit?",
    "What's the capital of France?",
]

for i, q in enumerate(test_questions, 1):
    result = ask_bot(q)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}")
    if result.get("sources"):
        print(
            f"Sources: "
            f"{[d.metadata.get('source') for d in result['sources']]}"
        )
    print("-" * 60)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

Batch Processing & Generating

In [ ]:
import pandas as pd

test_df = pd.read_csv("test.csv")
display(test_df)

answers = []
for i, row in test_df.iterrows():
    q = row["question"]
    result = ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/{len(test_df)}] {q}\n-> {result['answer']}\n{'-'*60}")

test_df["answer"] = answers

submission = test_df[["question_id", "answer"]]
submission.to_csv("submission.csv", index=False)
submission.head()

In [15]:
import os
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
models = client.models.list()

print("Available models:")
for m in models.data:
    print("-", m.id)

Available models:
- allam-2-7b
- meta-llama/llama-prompt-guard-2-86m
- canopylabs/orpheus-arabic-saudi
- canopylabs/orpheus-v1-english
- openai/gpt-oss-20b
- qwen/qwen3.8-27b
- openai/gpt-oss-safeguard-20b
- whisper-large-v3
- whisper-large-v3-turbo
- openai/gpt-oss-120b
- meta-llama/llama-prompt-guard-2-22m
